# How to swap between both providers running locally

Objective: Sanity check that loading / unloading works in quickish succession.

Steps

1. Run model using ollama
2. Unload all ollama models
3. Run model using llama-swap
4. Unload all llama-swap models
5. Run model using ollama
6. Unload all models

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import httpx

from fintl.common import Config, LlamaSwapConfig, OllamaConfig, Provider, Sources
from fintl.common.extraction.availability import check_llama_swap_ok, check_ollama_ok
from fintl.common.extraction.constants import LLAMA_SWAP_BASE_URL, OLLAMA_BASE_URL, ModelProvider
from fintl.common.extraction.unload import unload_llama_swap, unload_ollama
from fintl.common.logging import Logging

In [ ]:
TESTS_DIR = Path("../tests")
assert TESTS_DIR.exists()
assert TESTS_DIR.is_dir()

In [ ]:
FILES_ROOT_DIR = TESTS_DIR / "files/artefacts/Scalable-Capital"
assert FILES_ROOT_DIR.exists()
assert FILES_ROOT_DIR.is_dir()

In [ ]:
TARGET_DIR = Path("/tmp/ollama-llama-swap-switch-nb")
TARGET_DIR.mkdir(exist_ok=True)

In [ ]:
LOGGER_CONFIG_PATH = TESTS_DIR / "logger-config.json"
assert LOGGER_CONFIG_PATH.exists()
assert LOGGER_CONFIG_PATH.is_file()

In [ ]:
OLLAMA_MODEL = "qwen3.6:latest"
LLAMA_SWAP_MODEL = "qwen-3.6-27b"

OLLAMA_PROVIDER = ModelProvider.ollama
LLAMA_SWAP_PROVIDER = ModelProvider.llama_swap

In [ ]:
ollama_config = Config(
    target_dir=TARGET_DIR,
    sources=Sources(scalable=Provider(broker=FILES_ROOT_DIR)),
    logging=Logging(config_file=LOGGER_CONFIG_PATH),
    ollama=OllamaConfig(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL),
    model_provider=OLLAMA_PROVIDER,
)

In [ ]:
llama_swap_config = Config(
    target_dir=TARGET_DIR,
    sources=Sources(scalable=Provider(broker=FILES_ROOT_DIR)),
    logging=Logging(config_file=LOGGER_CONFIG_PATH),
    llama_swap=LlamaSwapConfig(model=LLAMA_SWAP_MODEL, base_url=LLAMA_SWAP_BASE_URL),
    model_provider=LLAMA_SWAP_PROVIDER,
)

## 1. Run model using ollama

In [ ]:
assert check_ollama_ok(ollama_config.ollama)

In [ ]:
_ollama_url = OLLAMA_BASE_URL
_ollama_url

In [ ]:
_ollama_generate_url = f"{_ollama_url}/api/generate"
_ollama_generate_url

In [ ]:
r = httpx.post(
    _ollama_generate_url,
    json={"model": OLLAMA_MODEL, "prompt": "Please say hello in some language.", "stream": False},
    timeout=ollama_config.model_timeout,
)
r.status_code

In [ ]:
print(r.json()["response"])

## 2. Unload ollama models

In [ ]:
unload_ollama(ollama_config.ollama, ollama_config.model_timeout)

## 3. Run model using llama-swap

In [ ]:
assert check_llama_swap_ok(llama_swap_config, do_inference_check=True)

## 4. Unload llama-swap models

In [ ]:
unload_llama_swap(llama_swap_config.llama_swap, llama_swap_config.model_timeout)

## 5. Run model using ollama

In [ ]:
r = httpx.post(
    _ollama_generate_url,
    json={"model": OLLAMA_MODEL, "prompt": "Please say hello in some language.", "stream": False},
    timeout=ollama_config.model_timeout,
)
r.status_code

In [ ]:
print(r.json()["response"])

## 6. Unload all models

In [ ]:
unload_ollama(ollama_config.ollama, ollama_config.model_timeout)

In [ ]:
unload_llama_swap(llama_swap_config.llama_swap, llama_swap_config.model_timeout)